# Visual Odometry

In [ ]:
#2 image merge

import cv2
import numpy as np

image1 = cv2.imread('image2.jpg')
image2 = cv2.imread('image1.jpg')

gray1 = cv2.cvtColor(image1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(image2, cv2.COLOR_BGR2GRAY)

orb = cv2.ORB_create()
keypoints1, descriptors1 = orb.detectAndCompute(gray1, None)
keypoints2, descriptors2 = orb.detectAndCompute(gray2, None)

#feature macthing
matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = matcher.match(descriptors1, descriptors2)

#good matches
good_matches = [match for match in matches if match.distance < 0.7 * max(len(matches), 1)]

src_pts = [keypoints1[match.queryIdx].pt for match in good_matches]
dst_pts = [keypoints2[match.trainIdx].pt for match in good_matches]

src_pts = np.array(src_pts, dtype=np.float32)
dst_pts = np.array(dst_pts, dtype=np.float32)

# Homography matrix
homography, _ = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC)

#Merge images
stitched_image = cv2.warpPerspective(image1, homography, (image1.shape[1] + image2.shape[1], image1.shape[0]))
stitched_image[0:image2.shape[0], 0:image2.shape[1]] = image2

threshold = 1
mask = cv2.inRange(stitched_image, (0, 0, 0), (threshold, threshold, threshold))
mask_inv = cv2.bitwise_not(mask)
coords = cv2.findNonZero(mask_inv)
x, y, w, h = cv2.boundingRect(coords)
stitched_image = stitched_image[y:y+h, x:x+w]
cv2.imshow('Cropped Image', stitched_image)
cv2.imwrite('merged_image.jpg',stitched_image)
cv2.waitKey(0)
cv2.destroyAllWindows()

# YOLO + Coordinate Assignment + A* + Exponential Effect Orientation

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import cvzone
import heapq
import random

# A* Variables
ROAD_VAL = 0
BARRIER_VAL = 1
FRIEND_VAL = 2
ENEMY_VAL = 3
FRIEND_INFLUENCE_VAL = 4
ENEMY_INFLUENCE_VAL = 5

ROAD_CLASS_ID = 4
FRIEND_CLASS_ID = 1
ENEMY_CLASS_ID = 2

# A* - Node Class
class Node:
    def _init_(self, x, y):
        self.x = x # row
        self.y = y # column
        self.g = float('inf')
        self.h = 0
        self.f = float('inf')
        self.parent = None
        self.blocked = False
        self.is_enemy_main = False
        self.is_friend_main = False
        self.is_enemy_influence = False
        self.is_friend_influence = False

    def _lt_(self, other):
        return self.f < other.f

    def _eq_(self, other):
        if not isinstance(other, Node):
            return NotImplemented
        return self.x == other.x and self.y == other.y

    def _hash_(self):
        return hash((self.x, self.y))

# Convert numerical value to character 
def get_char_representation(value):
    if value == ROAD_VAL:
        return '0'
    elif value == BARRIER_VAL:
        return '1'
    elif value == FRIEND_VAL or value == FRIEND_INFLUENCE_VAL:
        return '*'
    elif value == ENEMY_VAL or value == ENEMY_INFLUENCE_VAL:
        return '#'
    else:
        return '?'

# Grid reduction
def downscale_grid(grid_np, scale=2):
    h, w = grid_np.shape
    new_h, new_w = h // scale, w // scale
    downscaled = np.zeros((new_h, new_w), dtype=int)

    for i in range(new_h):
        for j in range(new_w):
            block = grid_np[i*scale:(i+1)*scale, j*scale:(j+1)*scale]
            
            if block.size == 0:
                downscaled[i, j] = BARRIER_VAL
                continue
            
            values, counts = np.unique(block, return_counts=True)
            downscaled[i, j] = values[np.argmax(counts)]
            
    return downscaled

# Function for Obtaining Neighboring Coordinates of the Domain
def get_neighbors_coords(x, y, rows, cols, radius=50):
    neighbors_set = set()
    for dx in range(-radius, radius + 1):
        for dy in range(-radius, radius + 1):
            nx, ny = x + dx, y + dy
            if 0 <= nx < rows and 0 <= ny < cols:
                neighbors_set.add((nx, ny))
    return neighbors_set

# Read Grid from a TXT File
def read_grid_from_text(filename):
    with open(filename, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]

    if not lines:
        raise ValueError("")

    first_row_chars = lines[0].split('Map file is empty or unreadable!')
    cols = len(first_row_chars)
    rows = len(lines)

    grid = [[Node(i, j) for j in range(cols)] for i in range(rows)]
    
    start_node_coord = None
    goal_node_coord = None

    for i in range(rows):
        current_row_chars = lines[i].split(' ')
        if len(current_row_chars) != cols:
            print(f"Warning: Row {i} does not match the expected number of columns ({cols}). It is being skipped or an error may occur.")
            continue

        for j in range(cols):
            char = current_row_chars[j]
            
            if char == get_char_representation(BARRIER_VAL):
                grid[i][j].blocked = True
            else:
                grid[i][j].blocked = False

            if char == get_char_representation(ENEMY_VAL):
                grid[i][j].is_enemy_influence = True
            elif char == get_char_representation(FRIEND_VAL):
                grid[i][j].is_friend_influence = True
            elif char == 'S': 
                start_node_coord = (i, j)
            elif char == 'G': 
                goal_node_coord = (i, j)
    
    penalties = ({"main": set(), "neighbors": set()}, {"main": set(), "neighbors": set()})

    return grid, (rows, cols), penalties, start_node_coord, goal_node_coord


# Map Visualization
def visualize_grid_and_effects(original_height, original_width, 
                               scaled_grid_for_astar, 
                               original_penalized_main, original_penalized_neighbors, 
                               original_preferred_main, original_preferred_neighbors, 
                               original_start_coord, original_goal_coord, path_original_scale=None):
    """It visualizes the map, areas of influence, start/bite points, and route in their original size."""
    
    grid_map_array = np.zeros((original_height, original_width, 3), dtype=np.uint8)
    
    grid_map_array[:, :] = [200, 200, 200] 

    rows_scaled_grid = len(scaled_grid_for_astar)
    cols_scaled_grid = len(scaled_grid_for_astar[0]) if rows_scaled_grid > 0 else 0

    downscale_factor_vis = original_height // rows_scaled_grid if rows_scaled_grid > 0 else 1 
                                                                          
    for r_scaled in range(rows_scaled_grid):
        for c_scaled in range(cols_scaled_grid):
            if scaled_grid_for_astar[r_scaled][c_scaled].blocked:
                for dr in range(downscale_factor_vis):
                    for dc in range(downscale_factor_vis):
                        r_orig = r_scaled * downscale_factor_vis + dr
                        c_orig = c_scaled * downscale_factor_vis + dc
                        if 0 <= r_orig < original_height and 0 <= c_orig < original_width:
                            grid_map_array[r_orig, c_orig] = [0, 0, 0] 

    for r, c in original_penalized_neighbors:
        if 0 <= r < original_height and 0 <= c < original_width:
            grid_map_array[r, c] = [255, 255, 0] 
    
    for r, c in original_preferred_neighbors:
        if 0 <= r < original_height and 0 <= c < original_width:
            if not (r, c) in original_penalized_neighbors: 
                grid_map_array[r, c] = [0, 255, 255] 

    for r, c in original_penalized_main:
        if 0 <= r < original_height and 0 <= c < original_width:
            grid_map_array[r, c] = [0, 0, 255]  
    for r, c in original_preferred_main:
        if 0 <= r < original_height and 0 <= c < original_width:
            grid_map_array[r, c] = [255, 0, 255] 

    if path_original_scale: 
        for r, c in path_original_scale:
            if 0 <= r < original_height and 0 <= c < original_width:
                if (r,c) != original_start_coord and (r,c) != original_goal_coord:
                    grid_map_array[r, c] = [255, 0, 0]

    if original_start_coord:
        start_r, start_c = original_start_coord 
        if 0 <= start_r < original_height and 0 <= start_c < original_width:
            grid_map_array[start_r, start_c] = [0, 255, 0]
    
    if original_goal_coord:
        goal_r, goal_c = original_goal_coord
        if 0 <= goal_r < original_height and 0 <= goal_c < original_width: 
            grid_map_array[goal_r, goal_c] = [0, 165, 255] 

    cv2.imshow("Map Image and Areas of Impact", grid_map_array)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    return grid_map_array

# Example A* Cost Calculation
def calculate_cost(current_node, neighbor_node):
    base_cost = 1 

    if neighbor_node.blocked:
        return float('inf')

    if neighbor_node.is_enemy_influence:
        return base_cost * 500 

    if neighbor_node.is_friend_influence:
        return base_cost * 0.5 

    return base_cost

# Implementation of the A* Algorithm
def astar_pathfinding(grid, start_coord, goal_coord, rows, cols):
    start_node = grid[start_coord[0]][start_coord[1]]
    goal_node = grid[goal_coord[0]][goal_coord[1]]

    if start_node.blocked:
        print(f"Error: Starting node ({start_coord}) is blocked. Path not found.")
        return None
    if goal_node.blocked:
        print(f"Error: Target node ({goal_coord}) is blocked. Path not found.")
        return None

    for r in range(rows):
        for c in range(cols):
            node = grid[r][c]
            node.g = float('inf')
            node.h = 0
            node.f = float('inf')
            node.parent = None

    open_list = []
    closed_list_set = set() 

    start_node.g = 0
    start_node.h = np.sqrt((start_node.x - goal_node.x)*2 + (start_node.y - goal_node.y)*2)
    start_node.f = start_node.g + start_node.h
    heapq.heappush(open_list, start_node)

    directions = [(0, 1), (0, -1), (1, 0), (-1, 0)] 

    while open_list:
        current_node = heapq.heappop(open_list)

        if (current_node.x, current_node.y) in closed_list_set: 
            continue
            
        closed_list_set.add((current_node.x, current_node.y))

        if current_node.x == goal_node.x and current_node.y == goal_node.y:
            path = []
            while current_node is not None:
                path.append((current_node.x, current_node.y))
                current_node = current_node.parent
            return path[::-1]


        for dr, dc in directions:
            neighbor_r, neighbor_c = current_node.x + dr, current_node.y + dc

            if not (0 <= neighbor_r < rows and 0 <= neighbor_c < cols):
                continue

            neighbor_node = grid[neighbor_r][neighbor_c]

            if (neighbor_r, neighbor_c) in closed_list_set or neighbor_node.blocked:
                continue
            
            move_cost_distance = np.sqrt(dr*2 + dc*2) 
            
            calculated_cost = calculate_cost(current_node, neighbor_node)

            new_g_cost = current_node.g + calculated_cost * move_cost_distance

            if new_g_cost < neighbor_node.g:
                neighbor_node.parent = current_node
                neighbor_node.g = new_g_cost
                neighbor_node.h = np.sqrt((neighbor_node.x - goal_node.x)*2 + (neighbor_node.y - goal_node.y)*2)
                neighbor_node.f = neighbor_node.g + neighbor_node.h
                
                heapq.heappush(open_list, neighbor_node)

    return None

# Main Process
def main():
    model_path = "best.pt"
    model = YOLO(model_path)

    img_path = "map.png"
    img = cv2.imread(img_path)
    if img is None:
        print(f"Error: The image '{img_path}' could not be loaded. Check the file path.")
        return

    original_height, original_width, channels = img.shape 

    print("YOLO model predictions are being launched...")
    results = model.predict(img, verbose=False)
    result = results[0]
    print("YOLO predictions are complete.")

    segmentation_masks_coordinates = []
    class_ids_for_masks = []

    if result.masks is not None:
        if hasattr(result.masks, 'segments') and result.masks.segments is not None:
            print("Segmented masks (segments) are being processed...")
            for i, seg in enumerate(result.masks.segments):
                seg_scaled = np.array(seg, dtype=np.int32)
                temp_mask = np.zeros((original_height, original_width), dtype=np.uint8)
                cv2.fillPoly(temp_mask, [seg_scaled], 1)
                mask_coords = np.column_stack(np.where(temp_mask > 0)) # (row, col)
                segmentation_masks_coordinates.append(mask_coords)
                if result.boxes is not None and hasattr(result.boxes, 'cls') and i < len(result.boxes.cls):
                    class_ids_for_masks.append(int(result.boxes.cls[i].cpu().item()))
                else:
                    class_ids_for_masks.append(-1)
        elif hasattr(result.masks, 'data') and result.masks.data is not None:
            print("Binary masks (data) is being processed...")
            for i, mask_tensor in enumerate(result.masks.data):
                mask = mask_tensor.cpu().numpy()
                mask_coordinates = np.column_stack(np.where(mask.astype(np.uint8) > 0))
                segmentation_masks_coordinates.append(mask_coordinates)
                if result.boxes is not None and hasattr(result.boxes, 'cls') and i < len(result.boxes.cls):
                    class_ids_for_masks.append(int(result.boxes.cls[i].cpu().item()))
                else:
                    class_ids_for_masks.append(-1)
    else:
        print("No mask data found.")

    road_segment_set = set()
    barrier_segment_set = set()
    friend_segment_set = set()
    enemy_segment_set = set()

    print("Segmented masks are being classified...")
    for i, mask_coords_list in enumerate(segmentation_masks_coordinates):
        if i < len(class_ids_for_masks):
            current_class_id = class_ids_for_masks[i] 
            for r, c in mask_coords_list:
                x, y = c, r # (col, row)
                if 0 <= x < original_width and 0 <= y < original_height:
                    if current_class_id == ROAD_CLASS_ID:
                        road_segment_set.add((x, y))
                    elif current_class_id == FRIEND_CLASS_ID:
                        friend_segment_set.add((x, y))
                    elif current_class_id == ENEMY_CLASS_ID:
                        enemy_segment_set.add((x, y))
                    else: 
                        barrier_segment_set.add((x, y))
        else: 
            for r, c in mask_coords_list:
                x, y = c, r
                if 0 <= x < original_width and 0 <= y < original_height:
                    barrier_segment_set.add((x, y))
    print("Segmentation classification is complete.")

    initial_grid_values = np.full((original_height, original_width), BARRIER_VAL, dtype=int)

    print("Orijinal grid değerleri atanıyor...")
    for i in range(original_height):
        for j in range(original_width):
            coord = (j, i) # (x, y) format
            if coord in enemy_segment_set:
                initial_grid_values[i, j] = ENEMY_VAL
            elif coord in friend_segment_set:
                initial_grid_values[i, j] = FRIEND_VAL
            elif coord in road_segment_set:
                initial_grid_values[i, j] = ROAD_VAL
    
    penalized_main_coords = set()
    for x_y in enemy_segment_set:
        penalized_main_coords.add((x_y[1], x_y[0])) # (row, col)

    preferred_main_coords = set()
    for x_y in friend_segment_set:
        preferred_main_coords.add((x_y[1], x_y[0])) # (row, col)

    penalized_neighbors = set()
    for r, c in penalized_main_coords:
        penalized_neighbors.update(get_neighbors_coords(r, c, original_height, original_width, radius=50))
    penalized_neighbors = penalized_neighbors - penalized_main_coords 

    preferred_neighbors = set()
    for r, c in preferred_main_coords:
        preferred_neighbors.update(get_neighbors_coords(r, c, original_height, original_width, radius=50))
    preferred_neighbors = preferred_neighbors - preferred_main_coords 

    final_txt_grid = np.copy(initial_grid_values) 

    for r, c in penalized_neighbors:
        if final_txt_grid[r, c] == ROAD_VAL: 
             final_txt_grid[r, c] = ENEMY_INFLUENCE_VAL 
    
    for r, c in preferred_neighbors:
        if final_txt_grid[r, c] == ROAD_VAL:
            final_txt_grid[r, c] = FRIEND_INFLUENCE_VAL 

    for r, c in penalized_main_coords:
        final_txt_grid[r, c] = ENEMY_VAL 
    
    for r, c in preferred_main_coords:
        final_txt_grid[r, c] = FRIEND_VAL 

    with open('output_x.txt', 'w') as f:
        for row in final_txt_grid:
            f.write(' '.join(map(get_char_representation, row)) + '\n')
    print("The file 'output_x.txt' has been saved.")

    downscale_factor = 4 
    print(f"The grid is reduced by a factor of {downscale_factor} and the file 'outputhalf{downscale_factor}.txt' is saved...")
    grid_small = downscale_grid(final_txt_grid, scale=downscale_factor)

    with open(f'outputhalf{downscale_factor}.txt', 'w') as f:
        for row in grid_small:
            f.write(' '.join(map(get_char_representation, row)) + '\n')
    print("İşlem tamamlandı.")

    try:
        grid_for_astar, (rows_astar, cols_astar), _, _, _ = read_grid_from_text(f'outputhalf{downscale_factor}.txt') 

        scaled_penalized_main_coords = set()
        for r, c in penalized_main_coords:
             scaled_penalized_main_coords.add((r // downscale_factor, c // downscale_factor))

        scaled_preferred_main_coords = set()
        for r, c in preferred_main_coords:
            scaled_preferred_main_coords.add((r // downscale_factor, c // downscale_factor))

        scaled_penalized_neighbors = set()
        for r, c in penalized_neighbors:
             scaled_penalized_neighbors.add((r // downscale_factor, c // downscale_factor))
        
        scaled_preferred_neighbors = set()
        for r, c in preferred_neighbors:
            scaled_preferred_neighbors.add((r // downscale_factor, c // downscale_factor))

        for r, c in scaled_penalized_main_coords:
            if 0 <= r < rows_astar and 0 <= c < cols_astar:
                grid_for_astar[r][c].is_enemy_main = True
                grid_for_astar[r][c].is_enemy_influence = True
        for r, c in scaled_preferred_main_coords:
            if 0 <= r < rows_astar and 0 <= c < cols_astar:
                grid_for_astar[r][c].is_friend_main = True
                grid_for_astar[r][c].is_friend_influence = True

        for r, c in scaled_penalized_neighbors:
            if 0 <= r < rows_astar and 0 <= c < cols_astar and not grid_for_astar[r][c].blocked:
                grid_for_astar[r][c].is_enemy_influence = True
        for r, c in scaled_preferred_neighbors:
            if 0 <= r < rows_astar and 0 <= c < cols_astar and not grid_for_astar[r][c].blocked:
                if not grid_for_astar[r][c].is_enemy_influence: 
                    grid_for_astar[r][c].is_friend_influence = True

        start_node_coord_txt = None
        goal_node_coord_txt = None

        def get_valid_coord(coord_name, rows_map, cols_map, grid_map, current_txt_map):
            while True:
                try:
                    row_str = input(f"Enter the line for {coord_name} (0-{rows_map-1}, reduced map size): ")
                    col_str = input(f"Enter the column for point {coord_name} (0-{cols_map-1}, reduced map size):")
                    
                    row = int(row_str)
                    col = int(col_str)
                    
                    if not (0 <= row < rows_map and 0 <= col < cols_map):
                        print("Error: The point is outside the map boundaries. Please try again.")
                        continue

                    if grid_map[row][col].blocked:
                        print(f"WARNING: The {coord_name} point ({row}, {col}) you entered is blocked (map value: '{get_char_representation(current_txt_map[row, col])}').")
                        print("Please select a point that is a path (0), enemy base/area of ​​influence (#) or friendly base/area of ​​influence (*).")
                        
                        suggestions = []
                        valid_chars_for_path = ['0', '#', '*']
                        
                        for _ in range(50): 
                            r_sugg = random.randint(0, rows_map - 1)
                            c_sugg = random.randint(0, cols_map - 1)
                            if not grid_map[r_sugg][c_sugg].blocked and get_char_representation(current_txt_map[r_sugg, c_sugg]) in valid_chars_for_path:
                                suggestions.append((r_sugg, c_sugg, get_char_representation(current_txt_map[r_sugg, c_sugg])))
                                if len(suggestions) >= 5: break
                        
                        if suggestions:
                            print("Suggested unblocked coordinates:")
                            for s_r, s_c, s_char in suggestions:
                                print(f"  - ({s_r}, {s_c}) (Değer: '{s_char}')")
                            print("You can try using one of the suggestions above.")
                        else:
                            print("Unfortunately, no suitable, unblocked coordinates were found. Please check your map.")
                            
                        continue
                    else:
                        print(f"The point {coord_name} ({row}, {col}) is not blocked. (Map value: '{get_char_representation(current_txt_map[row, col])}')")
                        return (row, col)
                except ValueError:
                    print("Invalid login. Please enter a number.")
        
        start_node_coord_txt = get_valid_coord("Beginning", rows_astar, cols_astar, grid_for_astar, grid_small)
        goal_node_coord_txt = get_valid_coord("Aim", rows_astar, cols_astar, grid_for_astar, grid_small)

        print(f"The A* algorithm ({start_node_coord_txt} -> {goal_node_coord_txt}) is being started (on the minimized map)...")
        
        path_found_scaled = astar_pathfinding(grid_for_astar, start_node_coord_txt, goal_node_coord_txt, rows_astar, cols_astar)
        
        path_found_original_scale = []
        if path_found_scaled:
            print(f"Path found! Scaled length: {len(path_found_scaled)} steps.")
            
            # Reduced path coordinates
            print("\nReduced Path Coordinates (row, col):")
            for coord in path_found_scaled:
                print(f"  {coord}")
            
            # Reduced path coordinates txt
            path_filename = "path_scaled_coords.txt"
            with open(path_filename, 'w') as f:
                for coord in path_found_scaled:
                    f.write(f"{coord}\n")
            print(f"The reduced path coordinates were saved to the '{path_filename}' file.")

            for r, c in path_found_scaled:
                original_r = r * downscale_factor + downscale_factor // 2
                original_c = c * downscale_factor + downscale_factor // 2
                path_found_original_scale.append((original_r, original_c))
        else:
            print("No route found. Check your starting/destination points or your map.")

        print("Maps and areas of influence are being visualized...")
        visualize_grid_and_effects(original_height, original_width, grid_for_astar, 
                                   penalized_main_coords, penalized_neighbors, 
                                   preferred_main_coords, preferred_neighbors,
                                   (start_node_coord_txt[0] * downscale_factor + downscale_factor // 2, 
                                    start_node_coord_txt[1] * downscale_factor + downscale_factor // 2), 
                                   (goal_node_coord_txt[0] * downscale_factor + downscale_factor // 2, 
                                    goal_node_coord_txt[1] * downscale_factor + downscale_factor // 2), 
                                   path_original_scale=path_found_original_scale) 

    except FileNotFoundError:
        print("Error: Map file not found. Please ensure the file is created after running the YOLO code.")
    except ValueError as e:
        print(f"Map reading error: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}. Please review the error message and the relevant section of the code.")

if _name_ == "_main_":
    main()

ModuleNotFoundError: No module named 'ultralytics'